Code taken from:
https://umap-learn.readthedocs.io/en/latest/basic_usage.html

Install the umap module with conda:
`conda install -c conda-forge umap-learn`

or with pip: 
`pip install umap-learn`

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
%matplotlib inline
import keras

import umap

from io import BytesIO
from PIL import Image
import base64

import json, sys
from scipy.io import  wavfile
from IPython import display

import str_ww_util as util
import get_dataset
import keras_model as models

from IPython import display

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource, CategoricalColorMapper
from bokeh.palettes import Spectral10
from bokeh.resources import INLINE

from get_dataset import get_data, get_file_lists, get_data_config

In [ ]:
output_notebook(resources=INLINE) # resource=INLINE needed for offline work
sns.set(style='white', context='notebook', rc={'figure.figsize':(14,10)})
rng = np.random.default_rng(2024)

# jupyter will pass an extra -f=<tmp_file> arg, which throws an 
# unrecognized argument error
sys.argv = sys.argv[0:1] 
Flags = util.parse_command("evaluate")

In [ ]:
notebook_mode = "short_inference" # "short_inference" OR "inference" OR "short_training" OR "full_training"

if notebook_mode == "inference": 
  load_pretrained_model = True
  save_model = False
elif notebook_mode == "short_inference": 
  load_pretrained_model = True
  save_model = False
  Flags.num_samples_training = 2000
  Flags.num_samples_validation = 1000
  Flags.num_samples_test = 1000
else:
  # Or make custom settings here
  pass

# 'trained_models/str_ww_model.h5' is the default save path for train.py
# pretrained_model_path = 'trained_models/str_ww_ref_model.h5' # path to load from if load_pretrained_model is True
pretrained_model_path = 'trained_models/str_ww_ref_model.h5' # path to load from if load_pretrained_model is True

samp_freq = Flags.sample_rate
label_list=['marvin', 'silent', 'other']

In [ ]:
try:
    with open('streaming_config.json', 'r') as fpi:
        streaming_config = json.load(fpi)
    Flags.data_dir = streaming_config['speech_commands_path']
except:
    raise RuntimeError("""
        In this directory, copy streaming_config_template.json to streaming_config.json
        and edit it to point to the directories where you have the speech commands dataset
        and (optionally) the MUSAN noise data set.
        """)
Flags.bg_path = Flags.data_dir

In [ ]:
ds_train, ds_test, ds_val = get_dataset.get_all_datasets(Flags)

In [ ]:
## This could probably be faster if we pre-allocate x_train, y_train  
## based on ds_train.cardinality() and then fill in the values.
x1, y1 = ds_train.as_numpy_iterator().next()
x_shape = x1.shape[1:]
y_shape = y1.shape[1:]
x_train = np.zeros((0,)+x_shape)
y_train = np.zeros((0,)+y_shape)
for x,y in ds_train:
  x_train = np.concatenate((x_train,x), axis=0)
  y_train = np.concatenate((y_train,y), axis=0)

y_train = np.argmax(y_train, axis=1) # convert from one hot to index

In [ ]:
print(f"Loading pretrained model from {pretrained_model_path}")
model = keras.models.load_model(pretrained_model_path)
model.summary()

In [ ]:


def get_features(x):
  use_features = True
  feature_layer = -4
  
  specgrams = x.squeeze()
  if use_features:
    feature_model = keras.Model(inputs=model.input, outputs=model.layers[feature_layer].output)
    features = feature_model(x)
    features = features.numpy()
    print(f"Using layer {feature_layer}, {model.layers[feature_layer].name},",
          f"type {model.layers[feature_layer].__class__.__name__}")
  else:
    print(f"Using input spectrograms")
    features = specgrams
  
  features = features.reshape(features.shape[0], -1)
  return features

feature_data = get_features(x_train)

In [ ]:
reducer = umap.UMAP(random_state=42)
reducer.fit(feature_data)

In [ ]:
embedding = reducer.transform(feature_data)
# Verify that the result of calling transform is
# idenitical to accessing the embedding_ attribute
assert(np.all(embedding == reducer.embedding_))
embedding.shape

In [ ]:
plt.figure(figsize=(8,8))
num_samples = x_train.shape[0]
idx = slice(0,num_samples)
plt.scatter(embedding[idx, 0], embedding[idx, 1], c=y_train[idx], cmap='Spectral', s=15)
plt.gca().set_aspect('equal', 'datalim')
hCBar = plt.colorbar(boundaries=np.arange(4)-0.5)
hCBar.set_ticks(np.arange(3))
hCBar.set_ticklabels(label_list)
plt.title('UMAP projection of the Keywords', fontsize=24);

In [ ]:
def embeddable_image(data):
    # scale from [0.0,1.0] => [0,255], invert (so large => black), and cast to uint8
    img_data = (255 * (1.0-data.T)).astype(np.uint8)
    image = Image.fromarray(img_data, mode='L').resize((64, 64), Image.BICUBIC)
    buffer = BytesIO()
    image.save(buffer, format='png')
    for_encoding = buffer.getvalue()
    return 'data:image/png;base64,' + base64.b64encode(for_encoding).decode()

# embeddable_image(specgrams[0]) => text encoding of an image, you can paste that into an html file.

In [ ]:
plt.figure(figsize=(5,5))
plt.imshow(specgrams[5].T, origin="lower", cmap="viridis")

In [ ]:
## Load the long wave and build the corresponding label waveform
wav_sampling_freq, long_wav = wavfile.read("long_wav.wav")
assert wav_sampling_freq == samp_freq
t = np.arange(len(long_wav))/samp_freq
print(f"min/max of long wav = {np.min(long_wav)} / {np.max(long_wav)}")
long_wav = long_wav / 2**15 # scale into [-1.0, +1.0] range

ww_windows_file = 'long_wav_ww_windows.json' # start/stop points of wakewords in a list
with open(ww_windows_file, 'r') as fpi:
  ww_windows = json.load(fpi)
  
# construct waveform that shows when wakeword is present
ww_present = np.zeros(len(long_wav))
for t_start, t_stop in ww_windows:
  idx_start = int(t_start*samp_freq)
  idx_stop  = int(t_stop*samp_freq)
  ww_present[idx_start:idx_stop] = 1

In [ ]:
## build a feature extractor that can operate on longer waveforms.
## this one can operate on waveforms up to len(long_wav)
data_config_val = get_dataset.get_data_config(Flags, 'validation')

with open("data_config_nb.json", "w") as fpo:
    json.dump(data_config_long, fpo, indent=4)
              
# long_wav = long_wav / np.max(np.abs(long_wav)) # scale into [-1.0, +1.0] range
t = np.arange(len(long_wav))/samp_freq

feature_extractor = get_dataset.get_lfbe_func(data_config_val)
# the feature extractor needs a label (in 1-hot format), but it doesn't matter what it is
# long_spec = feature_extractor_long({'audio':long_wav, 'label':[0.0, 0.0, 0.0]})['audio'].numpy()

# long_spec = feature_extractor_long(np.expand_dims(long_wav, 0)).numpy()
long_spec = feature_extractor(long_wav).numpy()

In [ ]:
plt.imshow(long_spec.T.squeeze()[:,200:1000])

In [ ]:
plt.figure(figsize=(5,2))
print(len(long_wav))
# idx = slice(265*16000,275*16000)
idx = slice(int(268.5*16000),int(269.5*16000))
plt.plot(t[idx], long_wav[idx], t[idx], ww_present[idx]-0.2)
plt.legend(['Waveform Clip', 'Wakeword present'])

In [ ]:
print(long_spec.shape)
print(long_wav.shape)
print(samp_freq)

In [ ]:
## Downsample ww_present so it aligns with the spectrogram

stride_samples = int(Flags.window_stride_ms*samp_freq/1000)
winlen_samples = int(Flags.window_size_ms*samp_freq/1000)
cliplen_samples = 16000
## ww_present_strided is repeated clips of winlen_samples(1024), strided by stride_samples (512)
## so it's the snippet that would go into one spectrogram calculation
ww_present_strided = np.lib.stride_tricks.sliding_window_view(ww_present, cliplen_samples)[0::stride_samples,:]

## Now determine whether each winlen_samples snippet is all negative  (=> 0), fully contains a positive (=>1)
## or partially contains a 1 (=> 0.5, and we'll leave it out of the dataset)
ww_present_subsampled = np.zeros(ww_present_strided.shape[0])
for i in range(len(ww_present_subsampled)):
  if ww_present_strided[i,0] == 1 or ww_present_strided[i,-1] == 1:
    # ww still in progress at beginning or end of clip, so not clear
    # if it should be labeled positive or negative
    ww_present_subsampled[i] = 0.5
  elif np.max(ww_present_strided[i,:]) == 1:
    ww_present_subsampled[i] = 1
  else:
    ww_present_subsampled[i] = 0


In [ ]:
## Show a clip of the spectrogram and corresponding wave and label
t_start = 200
t_stop = 205

plt.figure(figsize=(6,4))
wav_idx = slice(int(t_start*Flags.sample_rate),int(t_stop*Flags.sample_rate))

leftover_samples = int((Flags.window_size_ms - Flags.window_stride_ms)*Flags.sample_rate/1000)
t_wav= np.arange(wav_idx.start, wav_idx.stop)/Flags.sample_rate

# get the spectrogram clip corresponding to the wav clip 
# and downsample time to the spectrogram frame rate
spec_idx = slice(int(wav_idx.start/stride_samples), int(wav_idx.stop/stride_samples))
spec_clip = long_spec[spec_idx,:,:]
print(f"spec_clip.shape={spec_clip.shape}")
t_spec = t_wav[:-leftover_samples:stride_samples]

plt.subplot(3,1,1)
mels = np.arange(long_spec.shape[2])
plt.pcolormesh(t_spec, mels, spec_clip.squeeze().T, cmap="jet")
plt.xlim([t[wav_idx.start], t[wav_idx.stop]])

plt.subplot(3,1,2)
plt.plot(t[wav_idx], long_wav[wav_idx], t[wav_idx], ww_present[wav_idx]-0.2)
plt.xlim([t[wav_idx.start], t[wav_idx.stop]])

plt.tight_layout()

In [ ]:
# get chunks from the spectrogram the size of the NN's input tenosr
long_spec_strided = np.lib.stride_tricks.sliding_window_view(long_spec, 30, axis=0)
long_spec_strided = long_spec_strided.transpose(0,3,1,2) # rearrange into (batch, time, Y (singleton), channels)
long_spec_strided = long_spec_strided[:-1,:,:,:] # ww_present_subsampled is one shorter
print(f"{long_spec.shape} => {long_spec_strided.shape}")

In [ ]:
## filter out the ambiguous samples (ww_present_subsampled=0.5)
x_long = long_spec_strided[ww_present_subsampled!=0.5]
# ww_present ~ [0=ww absent, 1=ww present]
# label_list = ['marvin', 'silent', 'other'], so 
# 2*(1-ww_present) maps 0 (absent) =>2 (other); 1 (present) => 0 ('marvin'), 
# so we won't have any class 1 (silent) samples
y_long_dense = 2*(1-ww_present_subsampled[ww_present_subsampled!=0.5])
y_long_dense = y_long_dense.astype(int)

num_classes = model.output.shape[-1]
num_samples = x_long.shape[0]
y_long = np.zeros((num_samples, num_classes), dtype=int)
## Convert to 1-hot, Set the appropriate index in each row to 1
y_long[np.arange(num_samples), y_long_dense.astype(int)] = 1

print(f"{x_long.shape}, {y_long.shape}")

In [ ]:
out_long = model.predict(x_long)


In [ ]:
# 
ww_detected_long = (out_long[:,0]>0.95).astype(int) # np.argmax(out_long, axis=1)
precision = np.sum((ww_detected_long) * (y_long_dense == 0)) / np.sum(ww_detected_long)


In [ ]:
model.evaluate(x=x_long, y=y_long)
# model.input_shape

In [ ]:
feature_dat_long = get_features(x_long)
emb_long = reducer.transform(feature_dat_long)

data_df_long = pd.DataFrame(emb_long, columns=('x', 'y'))
#y_train[idx], label_list
data_df_long['class_name'] = [label_list[c] for c in y_long_dense]
data_df_long['image'] = list(map(embeddable_image, x_long.squeeze()))

In [ ]:
class_descriptors = ['Unset']*feature_dat_long.shape[0]

fps_long = np.nonzero(ww_detected_long*(1-y_long[:,0]))[0]
for i in fps_long: class_descriptors[i] = 'Long FP'

fns_long = np.nonzero((1-ww_detected_long)*(y_long[:,0]))[0]
for i in fns_long: class_descriptors[i] = 'Long FN'

tps_long = np.nonzero(ww_detected_long*(y_long[:,0]))[0]
for i in tps_long: class_descriptors[i] = 'Long TP'

tns_long = np.nonzero((1-ww_detected_long)*(1-y_long[:,0]))[0]
for i in tns_long: class_descriptors[i] = 'Long TN'



In [ ]:
data_df = pd.DataFrame(embedding, columns=('x', 'y'))
#y_train[idx], label_list
data_df['class_name'] = [label_list[c] for c in y_train]
data_df['image'] = list(map(embeddable_image, specgrams))

datasource = ColumnDataSource(data_df)
color_mapping = CategoricalColorMapper(factors=label_list,
                                       palette=["red", "green", "blue"])

plot_figure = figure(
    title='UMAP projection of Wakeword Features',
    width=600, 
    height=600,
    tools=(['pan', 'reset','wheel_zoom'])  # can replace wheel_zoom with zoom_in, zoom_out
)

plot_figure.add_tools(HoverTool(tooltips="""
<div>
    <div>
        <img src='@image' style='float: left; margin: 5px 5px 5px 5px'/>
    </div>
    <div>
        <span style='font-size: 16px; color: #224499'>Class:</span>
        <span style='font-size: 18px'>@class_name</span>
    </div>
</div>
"""))

plot_figure.scatter(
    'x',
    'y',
    source=datasource,
    color=dict(field='class_name', transform=color_mapping),
    line_alpha=0.6,
    fill_alpha=0.6,
    size=6
)
show(plot_figure)